In [ ]:
import os

from glob import glob
from PIL import Image

# from rich.pretty import pprint
from pydantic_ai import BinaryContent

from common.cache import RedisCache
from common.utils.path import create_path
from common.utils.json_data import save_json

from por.utils.tokens import get_num_tokens
from por.utils.prompt import get_drhdr_prompt
from por.multi_agent import get_multi_agent_config
from por.llm_agents import (
    ImageDescriber,
    ImageDescriberInput,
    ImageDescriberOutput,
)

In [ ]:
multi_agent_config = get_multi_agent_config()
MAX_IAMGE_PROMPT_TOKENS = 512

In [ ]:
def show_image(image_path: str) -> None:
    return Image.open(image_path)

In [ ]:
def get_binary_content(image_path: str) -> BinaryContent:
    with open(image_path, "rb") as image_file:
        return BinaryContent(
            data=image_file.read(),
            media_type="image/jpg",
        )

In [ ]:
def get_description(image_describer_output: ImageDescriberOutput) -> str:
    description = get_drhdr_prompt(
        physical_description=image_describer_output.physical_description,
        clothing_description=image_describer_output.clothing_description,
    )

    num_tokens = get_num_tokens(text=description)
    assert num_tokens <= MAX_IAMGE_PROMPT_TOKENS, print(
        f"{num_tokens} > {MAX_IAMGE_PROMPT_TOKENS}"
    )

    return description

In [ ]:
image_paths = glob("/resources/dorohedoro/train-images/*.jpg")
print(f"image_paths => {len(image_paths)}")

In [ ]:
agent_inputs = [
    ImageDescriberInput(
        description_guidelines=multi_agent_config.image_description_guidelines
    )
    for _ in image_paths
]

user_contents = [
    get_binary_content(image_path=image_path) for image_path in image_paths
]

image_describer = ImageDescriber(
    cache=RedisCache(),
    max_concurrency=5,
)

image_describer_outputs = await image_describer.batch_generate(
    agent_inputs=agent_inputs,
    user_contents=user_contents,
)

In [ ]:
# idx = 0

# pprint(image_describer_outputs[idx])
# Image.open(image_paths[idx])

In [ ]:
out_base_path = "/resources/dorohedoro/train-images"
for image_path, image_describer_output in zip(
    image_paths,
    image_describer_outputs,
):
    out_file = os.path.basename(image_path).replace(".jpg", ".txt")
    out_path = f"{out_base_path}/{out_file}"
    with open(out_path, "w") as file:
        description = get_description(
            image_describer_output=image_describer_output
        )

        file.write(description)


In [ ]:
image_captions = [
    {
        "image_path": image_path,
        "people_description": ido.physical_description,
        "scene_description": ido.clothing_description,
    }
    for image_path, ido in zip(image_paths, image_describer_outputs)
]

file_path = "/resources/data/train-image-captions.json"
create_path(os.path.dirname(file_path), overwrite=True)
save_json(obj=image_captions, file_path=file_path)